In [39]:
import pandas as pd
import xarray as xr
import numpy as np

In [40]:
def load_grid_info(country_csv, area_csv):
    country_map = pd.read_csv(country_csv)  # 必须包含 I,J,country
    area_df     = pd.read_csv(area_csv)     # 必须包含 I,J,以及面积列 Value

    grid_info = country_map.merge(area_df, on=["I", "J"], how="left")
    grid_info.rename(columns={"Value": "area"}, inplace=True)

    return grid_info

In [41]:
def build_country_area_matrices(grid_info, lat_size, lon_size):
    country_matrix = np.empty((lat_size, lon_size), dtype=object)
    area_matrix = np.zeros((lat_size, lon_size)) * np.nan

    for _, r in grid_info.iterrows():
        # 修正：GIJ 是 1-based，需要转为 python 0-based
        I = int(r["I"]) - 1   # lat index
        J = int(r["J"]) - 1   # lon index

        country_matrix[I, J] = r["country"]
        area_matrix[I, J]    = r["area"]

    return country_matrix, area_matrix


In [42]:
def compute_variable_area(ds, varname, year_list, ds_years, country_matrix, area_matrix):
    print(f"→ 处理变量 {varname}")

    da = ds[varname]
    da_fixed = da.transpose("time", "lat", "lon")

    results = []

    # 安全国家列表（关键修复）
    unique_countries = pd.unique(country_matrix.ravel())
    unique_countries = [
        c for c in unique_countries if c is not None and not pd.isna(c)
    ]

    # 将 area_matrix 中 NaN 变 0
    area_matrix = np.nan_to_num(area_matrix, nan=0.0)

    for year in year_list:

        t = np.where(ds_years == year)[0][0]

        frac = da_fixed[t].values
        frac = np.nan_to_num(frac, nan=0.0)  # 将 NaN 变 0

        real_area = frac * area_matrix

        for country in unique_countries:
            mask = (country_matrix == country)
            total = real_area[mask].sum()

            results.append([varname, country, year, total])

    return results


In [43]:
def compute_country_total_area(country_matrix, area_matrix, output_csv):
    """
    统计每个国家的总面积，并输出 CSV。
    """
    # 提取国家列表
    unique_countries = pd.unique(country_matrix.ravel())
    unique_countries = [
        c for c in unique_countries if c is not None and not pd.isna(c)
    ]

    # NaN 当 0
    area_matrix = np.nan_to_num(area_matrix, nan=0.0)

    results = []
    for country in unique_countries:
        mask = (country_matrix == country)
        total_area = area_matrix[mask].sum()
        results.append([country, total_area])

    df_area = pd.DataFrame(results, columns=["country", "total_area"])
    df_area.to_csv(output_csv, index=False)
    print(f"国家面积表已保存：{output_csv}")

    return df_area


In [44]:
def run_area_summary(name_list, year_list, PATH_COUNTRY, PATH_AREA, PATH_NC, OUTPUT_CSV):

    print("将处理的年份 =", year_list)
    print("将处理的变量 =", name_list)

    # --- 加载 grid 信息 ---
    grid_info = load_grid_info(PATH_COUNTRY, PATH_AREA)

    # --- 打开 NC 文件 ---
    ds = xr.open_dataset(PATH_NC)
    ds_years = pd.to_datetime(ds["time"].values).year

    lat_size = len(ds["lat"])
    lon_size = len(ds["lon"])

    # --- 构建矩阵 ---
    country_matrix, area_matrix = build_country_area_matrices(grid_info, lat_size, lon_size)
    
    PATH_COUNTRY_AREA = OUTPUT_CSV.replace(".csv", "_country_area.csv")
    df_country_area = compute_country_total_area(country_matrix, area_matrix, PATH_COUNTRY_AREA)

    # --- 循环变量 ---
    all_results = []

    for base_name in name_list:
        var_basin  = f"basin_{base_name}"
        var_region = f"region_{base_name}"

        for varname in [var_basin, var_region]:

            res = compute_variable_area(
                ds, varname, year_list, ds_years,
                country_matrix, area_matrix
            )
            all_results.extend(res)

    # --- 输出 CSV ---
    df = pd.DataFrame(all_results, columns=["variable", "country", "year", "area"])
    df.to_csv(OUTPUT_CSV, index=False)

    print(f"完成：结果已保存到 {OUTPUT_CSV}")

    return df


In [45]:
name_list = ["forest", "agri", "grassland"]    

year_list = list(range(2010, 2101, 10))
year_list.insert(0, 2005)

PATH_COUNTRY = "../../CSV/grid_country_output.csv"
PATH_AREA    = "../../CSV/GAIJ.csv"
PATH_NC      = "../../NC/compare.nc"
OUTPUT_CSV   = "../../CSV/country_area_timeseries.csv"
output_csv   = "../../CSV/countryarea.csv"

df = run_area_summary(name_list, year_list, PATH_COUNTRY, PATH_AREA, PATH_NC, OUTPUT_CSV)

将处理的年份 = [2005, 2010, 2020, 2030, 2040, 2050, 2060, 2070, 2080, 2090, 2100]
将处理的变量 = ['forest', 'agri', 'grassland']
国家面积表已保存：../../CSV/country_area_timeseries_country_area.csv
→ 处理变量 basin_forest
→ 处理变量 region_forest
→ 处理变量 basin_agri
→ 处理变量 region_agri
→ 处理变量 basin_grassland
→ 处理变量 region_grassland
完成：结果已保存到 ../../CSV/country_area_timeseries.csv


In [ ]:
grid_info[grid_info["area"] == 0]
